# Отбор признаков для uplift-модели (S-learner)

Витрина примерно на 4000 колонок с помесячной историей. Задача — отобрать небольшой
набор признаков для модели, предсказывающей отклик на push-кампанию.

Метод — **каскад из четырёх фильтров**, от дешёвых массовых к дорогим точным.
Порядок принципиален: последние два шага на четырёх тысячах признаков не запустятся.

| Этап | Что делает |
|---|---|
| Предобработка | пропуски, корреляции, типы, feature engineering |
| Сэмпл | Spark → pandas, дальше всё в памяти одной машины |
| Шаг 1 | важности LightGBM ∩ важности SHAP, каждая с пересечением по 5 фолдам |
| Шаг 2 | временная стабильность (adversarial validation по парам месяцев) |
| Шаг 3 | BorutaShap (сравнение с shadow-признаками) |
| Шаг 4 | рекурсивное исключение по SHAP + отсечка по кривой лосса |

Ноутбук исполняется строго сверху вниз.

## Импорты и константы

In [ ]:
import os
import sys

# ----------------COMMENT FOR DATALAB PRO----------------
os.environ['SPARK_MAJOR_VERSION'] = '3'
os.environ['SPARK_HOME'] = '/usr/sdp/current/spark3-client/'
os.environ['PYSPARK_PYTHON'] = '/data/sdp/mlpy3811v23/bin/python'
os.environ['PYSPARK_DRIVER_PYTHON'] = '/data/sdp/mlpy3811v23/bin/python'
os.environ['LD_LIBRARY_PATH'] = '/opt/python/virtualenv/jupyter/lib'
sys.path.insert(0, '/usr/sdp/current/spark3-client/python/')
sys.path.insert(0, '/usr/sdp/current/spark3-client/python/lib/py4j-0.10.9.3-src.zip')
sys.path.append('../')

In [ ]:
import re
import json
import random
import joblib

from IPython.display import clear_output
from tqdm import tqdm

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.style.use('ggplot')

from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report

from lightgbm import LGBMClassifier
from catboost import (CatBoostClassifier, CatBoostError,
                      EShapCalcType, EFeaturesSelectionAlgorithm)
from BorutaShap import BorutaShap

import pyspark.sql.functions as F
from pyspark.sql.types import IntegerType

from utils.get_spark import get_spark
from utils import show_results
from utils.feature_selection import *

%matplotlib inline

In [ ]:
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

PRODUCT = 'BO_PUSH'
PREFIX = 'srs'
DATASET_TABLE = f'{PREFIX}_{PRODUCT.lower()}_dataset'

target_column = 'target'
treatment_column = 'cg_flg'      # 1 = КОНТРОЛЬНАЯ группа
date_column = 'report_dt'        # последний день месяца

# Служебные поля витрины — не признаки
technical_columns = set([
    'cltv_lag_3', 'pl_val_lag_3', 'residual_npv_val_lag_3', 'sale_dt', 'start_dt',
    'target', 'channel_name', 'epk_id', 'res_1', 'id_target', 'ctl_loading',
    'row_upd_dttm', 'insert_time', 'ctl_validfrom', 'ctl_action', 'insert_dttm',
    'hash_val', 'row_upd_dt',
])

# Месяцы с недозревшим таргетом
dates_not_to_include = ['2025-09-30', '2025-10-31']

# Пороги пайплайна
NULL_THRESHOLD = 0.95        # выкидываем колонку, если пропусков >= 95%
CORR_THRESHOLD = 0.95        # выкидываем один признак из пары с |corr| > 0.95
CORR_SAMPLE_ROWS = 200_000   # на скольких строках считаем корреляции
SAMPLE_ROWS = 700_000        # бюджет строк для отбора признаков
IMP_THRESHOLD = 0.85         # порог по кумулятивной важности (шаг 1)
STABILITY_THRESHOLD = 0.6    # порог AUC разделения месяцев (шаг 2)
BORUTA_TRIALS = 50           # число трейлов BorutaShap (шаг 3)

CACHE_DIR = './cache/feature_selection/s'
IMG_DIR = './img/feature_selection'

os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(IMG_DIR, exist_ok=True)

## Загрузка

Из витрины сразу выбрасываются месяцы, где таргет ещё не дозрел.

In [ ]:
spark = get_spark(app_name='srs-bo-push-fs')
clear_output()

dataset = spark.table(f'arnsdpsbx_t_team_mp_cmpn_ds.{DATASET_TABLE}')
dataset = dataset.withColumn(date_column, F.col(date_column).cast('string'))
dataset = dataset.withColumn('epk_id', F.col('epk_id').cast('string'))

dates_target = sorted(row[date_column] for row in
                      dataset.select(date_column).distinct().collect())
dates_target = [d for d in dates_target if d not in dates_not_to_include]
dataset = dataset.filter(F.col(date_column).isin(dates_target))
clear_output()

print(f'Месяцы: {dates_target}')
print(f'Колонок на входе: {len(dataset.columns)}')

### Проверка данных

Смотрим глазами: отклик по месяцам и группам, пустые таргеты, дубли клиентов.
На данные не влияет.

In [ ]:
check_pd = dataset.select(['epk_id', date_column, treatment_column, target_column]).toPandas()
check_pd = check_pd.dropna(subset=['epk_id'])

print('Отклик по месяцам (столбцы — cg_flg):')
display(check_pd.groupby([date_column, treatment_column])[target_column].mean().unstack())

print(f'Строк с пустым таргетом: {check_pd[target_column].isna().sum():,}')
print(f'Дублей epk_id внутри месяца: {check_pd.duplicated(subset=["epk_id", date_column]).sum():,}')

del check_pd

## Предобработка

Четыре дешёвых фильтра подряд. Каждый работает на всей выборке в Spark и сокращает
число колонок до того, как начнётся дорогой отбор.

### 1. Пропуски

Считаем долю null по каждой колонке одним проходом и оставляем те, где её меньше 95%.

In [ ]:
def remove_null_cols(df, thr=NULL_THRESHOLD):
    """Оставляем колонки, в которых доля пропусков меньше thr."""
    df_len = df.count()
    null_stats = df.select([
        (F.count(F.when(F.isnull(c), c)) / df_len).alias(c) for c in df.columns
    ]).collect()[0].asDict()

    columns = [col for col, per_null in null_stats.items() if per_null < thr]
    return df.select(columns), null_stats


n_before = len(dataset.columns)
dataset, null_stats = remove_null_cols(dataset)
len_dataset = dataset.count()

print(f'Колонок: {n_before} → {len(dataset.columns)}')
print(f'Строк: {len_dataset:,}')

### 2. Корреляции

Жадный проход по матрице корреляций: если у пары признаков `|corr| > 0.95`, выбрасываем
второй по порядку. Какого именно из пары считать «вторым» — определяет порядок колонок,
критерия полезности здесь нет.

Матрица считается не на всей выборке, а на сэмпле в 200 тысяч строк — этого достаточно
для оценки корреляции, а полный расчёт на 4000 колонок был бы неподъёмным.

In [ ]:
# decimal → float, иначе корреляции не посчитаются
decimal_columns = [name for name, dtype in dataset.dtypes if 'decimal' in dtype]
for col_name in decimal_columns:
    dataset = dataset.withColumn(col_name, F.col(col_name).cast('float'))
print(f'decimal → float: {len(decimal_columns)} колонок')

In [ ]:
def find_correlated_columns(subset, thr=CORR_THRESHOLD):
    """Из каждой пары с |corr| > thr выбрасываем второй по порядку признак.

    Уже выброшенные не рассматриваются как «первые» — иначе выкинули бы обоих.
    """
    corrs = subset.corr(numeric_only=True)
    columns = corrs.columns

    correlated_columns = []
    correlated_pairs = {}

    for i in range(len(columns)):
        if columns[i] in correlated_columns:
            continue
        for j in range(i + 1, len(columns)):
            val = corrs.iloc[i, j]
            if abs(val) > thr:                     # и положительная, и отрицательная
                correlated_pairs[(columns[i], columns[j])] = val
                correlated_columns.append(columns[j])

    return set(correlated_columns), correlated_pairs

In [ ]:
columns_to_check = [name for name, dtype in dataset.dtypes
                    if dtype not in ('string', 'timestamp')]
columns_to_check = list(set(columns_to_check) - technical_columns
                        - {target_column, treatment_column, date_column})

fraction = min(float(np.round(CORR_SAMPLE_ROWS / len_dataset, 2)), 1.0)
check_corr_dataset = (dataset.select(columns_to_check)
                      .sample(fraction=fraction, withReplacement=False, seed=SEED)
                      .toPandas())
print(f'Сэмпл для корреляций: {check_corr_dataset.shape}')

correlated_columns, corr_stats = find_correlated_columns(check_corr_dataset)
print(f'Выброшено по корреляции: {len(correlated_columns)}')

del check_corr_dataset

In [ ]:
features = (set(dataset.columns) - technical_columns - correlated_columns
            - {target_column, treatment_column, date_column})
dataset = dataset.select(list(features) + [target_column, treatment_column, date_column])

joblib.dump(features, f'{CACHE_DIR}/features_after_base_prep.pkl')
joblib.dump(corr_stats, f'{CACHE_DIR}/corr_stats.pkl')

print(f'Признаков после фильтров: {len(features)}')

### 3. Приведение типов

Три вида колонок бустинги не понимают как есть:

* **даты** — вместо самой даты считаем число дней между ней и `report_dt` (суффикс `_diff`);
* **битовые маски** вида `'110101X'` — складываем единицы (суффикс `_sum`, символы `X` игнорируем);
* **строковые** — просто запоминаем список, кодировать будем после сэмпла.

In [ ]:
def count_dt_diff(sdf, dt_features, date_col):
    """Датовые признаки → число дней между report_dt и этой датой."""
    for col in dt_features:
        sdf = sdf.withColumn(col + '_diff', F.datediff(F.col(date_col), F.col(col))).drop(col)
    return sdf


@F.udf(returnType=IntegerType())
def count_ones(col):
    """Битовая маска → сумма единиц. Бывают null и символы 'X'."""
    if isinstance(col, str):
        return sum(int(e) for e in col if e.isdigit())


def count_payroll(sdf, bitmask_features):
    """Битовые маски → их сумма."""
    for col in bitmask_features:
        sdf = sdf.withColumn(col + '_sum', count_ones(col)).drop(col)
    return sdf

In [ ]:
dt_features = [c for c in dataset.columns
               if re.search(r'.*(_dt).*', c) and c != date_column]
dataset = count_dt_diff(dataset, dt_features, date_column)

bitmask_cols = [c for c in dataset.columns if 'bitmask' in c]
dataset = count_payroll(dataset, bitmask_cols)

string_features = [name for name, dtype in dataset.dtypes
                   if dtype == 'string' and name not in (target_column, date_column)]

print(f'Датовых: {len(dt_features)}, битовых масок: {len(bitmask_cols)}, '
      f'строковых: {len(string_features)}')

### 4. Пропуски в значениях

Строки без таргета удаляем — их нечем заменить. Остальное заполняем сентинелами:
для деревьев это нормально, модель выделит `-999999` в отдельную ветку и тем самым
сама выучит признак «здесь было пусто».

In [ ]:
print(f'Строк с пустым таргетом: {dataset.filter(F.isnull(target_column)).count():,}')

dataset = dataset.dropna(subset=[target_column])
dataset = dataset.fillna(-999999)   # числовые
dataset = dataset.fillna('')        # строковые

len_dataset = dataset.count()
print(f'Строк после очистки: {len_dataset:,}')

## Сэмпл в pandas

**Ключевая точка пайплайна.** Дальше Spark не используется: все четыре метода отбора
работают в памяти одной машины на этом сэмпле.

Страта — склейка `таргет_группа_месяц`, и из каждой страты берётся одинаковая доля.
Это не балансировка: дисбаланс классов сохраняется, гарантируется только то, что все
комбинации будут представлены.

In [ ]:
if len_dataset < SAMPLE_ROWS:
    dataset_pd = dataset.toPandas()
else:
    fraction = float(np.round(SAMPLE_ROWS / len_dataset, 2))

    dataset = dataset.withColumn('stratification_col', F.concat(
        F.col(target_column).cast('string'), F.lit('_'),
        F.col(treatment_column).cast('string'), F.lit('_'),
        F.col(date_column).cast('string')))

    strata = [row['stratification_col'] for row in
              dataset.select('stratification_col').distinct().collect()]

    dataset_pd = dataset.sampleBy('stratification_col',
                                  fractions={s: fraction for s in strata},
                                  seed=SEED).toPandas()
    dataset_pd = dataset_pd.drop(columns=['stratification_col'])
    print(f'Доля сэмпла: {fraction}')

print(f'Сэмпл: {dataset_pd.shape}')
print(f'Отклик по группам:\n{dataset_pd.groupby(treatment_column)[target_column].mean()}')

### Кодирование и treatment

Категории кодируются порядковыми номерами. Номер произвольный — нативная работа
бустингов с категориями здесь не используется.

`cg_flg == 1` означает **контрольную** группу, поэтому `is_treated` — это инверсия
флага. Дальше он участвует в отборе наравне с обычными признаками, а выбрасывается
только в самом конце.

In [ ]:
for col in string_features:
    dataset_pd[col] = dataset_pd[col].astype('category')

enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=np.nan)
dataset_pd[string_features] = enc.fit_transform(dataset_pd[string_features])

dataset_pd['is_treated'] = dataset_pd[treatment_column].map({1: 0, 0: 1})

# имена признаков изменились после count_dt_diff / count_payroll
prep_features = []
for feat in features:
    if feat in dt_features:
        prep_features.append(feat + '_diff')
    elif feat in bitmask_cols:
        prep_features.append(feat + '_sum')
    else:
        prep_features.append(feat)

dataset_pd = dataset_pd[prep_features + ['is_treated', target_column, date_column]]
print(f'Признаков на вход отбора: {len(prep_features)}')

## Измеритель качества

Не часть отбора — ничего из него не возвращается в пайплайн. Нужен, чтобы ответить,
сколько качества теряется при сокращении числа признаков, поэтому вызывается дважды:
сейчас на всех признаках (эталон) и в конце на отобранных.

Меряет две вещи:

* **классификацию** — ROC AUC и classification report;
* **аплифт** — для S-модели это разность предсказаний одной модели при `is_treated=1`
  и `is_treated=0`, дальше Qini-кривая и uplift@k через `show_results`.

In [ ]:
def evaluate(features_list, save_prefix):
    """Учим S-модель на наборе признаков и меряем классификацию и аплифт."""
    cols = [f for f in features_list if f != 'is_treated']

    train, val = train_test_split(
        dataset_pd, test_size=0.2, shuffle=True,
        stratify=dataset_pd[['is_treated', target_column]], random_state=SEED)

    model = CatBoostClassifier(iterations=700, verbose=100, random_state=SEED)
    model.fit(train[cols + ['is_treated']], train[target_column])

    preds = model.predict_proba(val[cols + ['is_treated']])[:, 1]
    print(f'\nROC AUC: {roc_auc_score(val[target_column], preds):.4f}')
    print(classification_report(val[target_column], (preds >= 0.1).astype(int)))

    # аплифт = P(отклик | коммуникация) - P(отклик | без коммуникации)
    uplift = (model.predict_proba(val[cols].assign(is_treated=1))[:, 1] -
              model.predict_proba(val[cols].assign(is_treated=0))[:, 1])

    metrics = show_results(val[target_column].values.ravel(), uplift,
                           val['is_treated'].values.ravel(),
                           save=True, save_path=IMG_DIR, save_prefix=save_prefix)
    return metrics

In [ ]:
metrics_all = evaluate(prep_features, save_prefix='all_features')

## Шаг 1 — важности LightGBM и SHAP

Два селектора из `utils/feature_selection.py`, устроены одинаково: 5-фолдовый
`StratifiedKFold`, на каждом фолде обучается LightGBM, важности считаются на
валидационной части фолда. Разница только в мере важности — число сплитов у первого
и среднее `|SHAP|` у второго.

Здесь неочевидное. `get_selected_features(0.85)` — это **не** «взять топ-85% важности».
Внутри топ по кумулятивной важности берётся отдельно для каждого фолда, и пять
получившихся списков **пересекаются**: признак обязан попасть в топ во всех пяти.
Сверху накладывается пересечение двух моделей. Итого пересечение десяти множеств —
именно оно даёт основное сужение, а не сам порог.

In [ ]:
x = dataset_pd.drop(columns=[target_column, date_column])
y = dataset_pd[target_column]

lgm_fs = LGMFeatureSelection(random_state=SEED)
lgm_fs.fit(x=x.values, y=y.values, columns=x.columns.tolist())
lgm_fs.importances.to_csv(f'{CACHE_DIR}/lgm_fs_imps.csv', index=False)

shap_fs = ShapFeatureSelection(random_state=SEED)
shap_fs.fit(x=x.values, y=y.values, columns=x.columns.tolist())
shap_fs.importances.to_csv(f'{CACHE_DIR}/shap_fs_imps.csv', index=False)

del x, y
clear_output()

In [ ]:
lgm_selected = lgm_fs.get_selected_features(IMP_THRESHOLD)
shap_selected = shap_fs.get_selected_features(IMP_THRESHOLD)
selected = list(set(lgm_selected) & set(shap_selected))

joblib.dump(selected, f'{CACHE_DIR}/selected_step1.pkl')

print(f'LGBM: {len(lgm_selected)}, SHAP: {len(shap_selected)}')
print(f'{len(prep_features)} → {len(selected)}')

## Шаг 2 — временная стабильность

Adversarial validation. Для каждой пары месяцев и каждого признака строится
искусственная задача: угадать по одному этому признаку, из какого месяца строка.
Высокий AUC означает, что распределение признака между месяцами разъехалось.

Отбор обратный интуиции: оставляем признаки с AUC **меньше** 0.6, то есть те, которые
месяцы различить **не могут** — они стабильны во времени и не подведут на новых данных.

Стоимость — `признаков × пар месяцев` обучений, поэтому шаг и стоит после первого.

In [ ]:
def check_feature_stability(df, date_col, dates, features_list):
    """Для каждой пары месяцев учим модель различать их по одному признаку.

    Возвращает {признак: [AUC по каждой паре месяцев]}.
    """
    results = {feature: [] for feature in features_list}
    combs = []

    for i, first_date in enumerate(dates):
        for second_date in dates[i + 1:]:
            combs.append((first_date, second_date))

            for feature in tqdm(features_list, desc=f'{first_date} vs {second_date}'):
                check_df = df[df[date_col].isin([first_date, second_date])][[date_col, feature]]
                is_second = (check_df[date_col] == second_date).astype(int)

                x_train, x_test, y_train, y_test = train_test_split(
                    check_df[[feature]], is_second, test_size=0.33, random_state=SEED)
                try:
                    estimator = CatBoostClassifier(n_estimators=500, max_depth=5,
                                                   eval_fraction=0.2, od_wait=15,
                                                   random_state=SEED, verbose=False)
                    estimator.fit(x_train, y_train)
                    score = roc_auc_score(y_test, estimator.predict_proba(x_test)[:, 1])
                except CatBoostError as e:
                    print(f'Feature: {feature}\n{e}')
                    score = 0.5

                results[feature].append(score)

    return results, combs

In [ ]:
dates = sorted(dataset_pd[date_column].unique())
stability_results, stability_combs = check_feature_stability(
    dataset_pd, date_column, dates, selected)

joblib.dump(stability_results, f'{CACHE_DIR}/stability_results.pkl')
clear_output()

mean_auc = pd.DataFrame(stability_results).mean().sort_values(ascending=False)
selected = mean_auc[mean_auc < STABILITY_THRESHOLD].index.tolist()
joblib.dump(selected, f'{CACHE_DIR}/selected_step2.pkl')

print(f'Отброшены как дрейфующие ({(mean_auc >= STABILITY_THRESHOLD).sum()}):')
display(mean_auc[mean_auc >= STABILITY_THRESHOLD])
print(f'→ {len(selected)}')

## Шаг 3 — BorutaShap

К каждому признаку добавляется его «тень» — та же колонка со случайно перемешанными
значениями. Если настоящий признак систематически важнее своих теней на 50 повторах,
он принимается. `TentativeRoughFix()` добивает те, по которым статистики не набралось.

Единственный шаг каскада со статистическим критерием, а не с порогом на глаз.

In [ ]:
model = LGBMClassifier(max_depth=5, n_estimators=500, learning_rate=0.05,
                       verbose=-100, random_state=SEED)

boruta_fs = BorutaShap(model=model, importance_measure='shap', classification=True)
boruta_fs.fit(X=dataset_pd[selected], y=dataset_pd[target_column],
              n_trials=BORUTA_TRIALS, random_state=SEED)
boruta_fs.TentativeRoughFix()

joblib.dump(boruta_fs, f'{CACHE_DIR}/boruta_fs.pkl')

n_before = len(selected)
selected = list(set(boruta_fs.accepted))
joblib.dump(selected, f'{CACHE_DIR}/selected_step3.pkl')

print(f'{n_before} → {len(selected)}')

## Шаг 4 — рекурсивное исключение по SHAP

CatBoost выбивает признаки по одному, каждый раз пересчитывая SHAP. Параметры выбраны
так, чтобы дойти до последнего признака — не потому что нужен один, а чтобы получить
полную кривую ухудшения лосса и выбрать отсечку по ней.

In [ ]:
x = dataset_pd[selected]
y = dataset_pd[target_column]
x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=0.3, random_state=SEED)

cb = CatBoostClassifier(iterations=1000, random_state=SEED)
summary = cb.select_features(
    x_train, y_train,
    eval_set=(x_val, y_val),
    features_for_select=x_train.columns.tolist(),
    num_features_to_select=1,               # выбиваем до последнего ради кривой
    steps=x_train.shape[1] - 1,
    algorithm=EFeaturesSelectionAlgorithm.RecursiveByShapValues,
    shap_calc_type=EShapCalcType.Regular,
    train_final_model=False,
    logging_level='Silent',
    plot=False,
)

with open(f'{CACHE_DIR}/summary_catboost.json', 'w') as f:
    json.dump(summary, f)

del x, y, x_train, x_val, y_train, y_val

### Отсечка по кривой

По горизонтали — признаки в порядке выбивания, по вертикали — лосс. Признаки, выбитые
**позже**, важнее. Нужно найти точку, где лосс начал устойчиво расти, и вписать имя
признака из этой точки в `CUT_FEATURE` в следующей ячейке — всё, что выбито начиная
с неё, забирается обратно.

Отсечка ручная: порог задаётся именем признака, поэтому на другом датасете его надо
выбирать заново по этому же графику.

In [ ]:
losses = summary['loss_graph']['loss_values'][1:]
eliminated = summary['eliminated_features_names']

plt.figure(figsize=(22, 8))
plt.plot(range(len(losses)), losses)
plt.xticks(range(len(eliminated)), eliminated, rotation=90)
plt.title('Лосс по мере исключения признаков (слева выбиты первыми)')
plt.tight_layout()
plt.savefig(f'{IMG_DIR}/recursive_selection_s.png', bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
CUT_FEATURE = 'tag_40590'      # <-- имя признака с графика выше

cut = eliminated.index(CUT_FEATURE)
selected = summary['selected_features_names'] + eliminated[cut:]

# is_treated участвовал в отборе наравне с признаками — теперь убираем
selected_all = [f for f in selected if f != 'is_treated']
joblib.dump(selected_all, f'{CACHE_DIR}/features_all.pkl')

print(f'Отсечка на шаге {cut} из {len(eliminated)}')
print(f'Итого признаков: {len(selected_all)}')
selected_all

## Результат

Второй прогон измерителя — теперь на отобранных признаках. Сравниваем с эталоном
на всех признаках: сколько аплифта потеряли, сократившись в сотни раз.

In [ ]:
metrics_selected = evaluate(selected_all, save_prefix='selected_features')

In [ ]:
comparison = pd.concat([
    metrics_all.assign(features=f'все ({len(prep_features)})'),
    metrics_selected.assign(features=f'отобранные ({len(selected_all)})'),
]).set_index('features')

comparison

## Описание отобранных признаков

Отчётный блок: подтягиваем расшифровки из справочника и рисуем таблицу картинкой.
Имена приводятся к виду справочника — снимаются суффиксы `_diff`/`_sum`, префикс `tag_`,
эмбеддинги схлопываются к общему имени.

In [ ]:
megav_desc = pd.read_excel('./cache/feature_description.xlsx', sheet_name='megav')
hashtags_desc = pd.read_excel('./cache/feature_description.xlsx', sheet_name='hashtags')
risk_desc = pd.read_excel('./cache/feature_description.xlsx', sheet_name='risk')

feature_descriptions = pd.concat([megav_desc, hashtags_desc, risk_desc], ignore_index=True)
feature_descriptions['name'] = feature_descriptions['name'].astype('str')

feature_descriptions = pd.concat([feature_descriptions, pd.DataFrame({
    'name': ['seq_hidden_state', 'embedding'],
    'description': ['Эмбеддинг ЦД CDS', 'Эмбеддинг ML360'],
    'tag_chld': [np.nan, np.nan]})], ignore_index=True)

del megav_desc, hashtags_desc, risk_desc

In [ ]:
def to_reference_name(name):
    """Имя признака → имя в справочнике."""
    if name.endswith('_diff'):
        name = name[:-5]
    if name.startswith('tag_'):
        name = name[4:]
    if name.endswith('_sum'):
        name = name[:-4]
    if name.startswith('embedding_'):
        name = name[:9]
    if name.startswith('seq_h'):
        name = name[:name.index('state_') + 5]
    return name


selected_all_df = pd.DataFrame({'feature': selected_all})
selected_all_df['name'] = selected_all_df['feature'].apply(to_reference_name)

f_desc = selected_all_df.merge(feature_descriptions, on='name', how='left')
f_desc.to_excel(f'{CACHE_DIR}/selected_feature_description.xlsx', index=False)
f_desc

In [ ]:
pd.set_option('display.max_colwidth', 1000)

fig, ax = plt.subplots(figsize=(12, 3))
ax.axis('tight')
ax.axis('off')

table = ax.table(cellText=f_desc.values, colLabels=f_desc.columns,
                 cellLoc='center', loc='center')
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(2.5, 1.8)

plt.savefig(f'{IMG_DIR}/feature_description_s.png', bbox_inches='tight', dpi=300)
plt.show()